# The Ising model of a magnet, by Metropolis sampling

The Ising model places a spin, either +1 or -1, on every site of a square lattice whose edges
wrap around. Each pair of neighbouring spins `s` and `t` contributes `-s * t` to the energy, so
that neighbours which agree lower it. At inverse temperature `beta`, the Metropolis method
proposes to flip a spin and accepts the flip with probability `min(1, exp(-beta * dE))`, where
`dE` is the change that the flip would make to the energy. After many flips, the lattices that
the method visits occur with the probabilities that statistical mechanics predicts. At low
temperature the spins line up and the magnetisation is large. At high temperature the spins are
mixed and the magnetisation is small.

The function below updates the sites of a chessboard in two halves, first the black squares and
then the white squares. No two sites of the same colour are neighbours, so all the sites of one
colour can be updated at once. A full update of the lattice is a sweep. The function starts with
every spin up and runs some sweeps to reach equilibrium. It then runs more sweeps, and after
each one it measures the energy and the absolute magnetisation, both per spin. It returns the
averages of the measurements. The demonstration prints the averages for a lattice of side 16 at
a high and at a low temperature.

In [ ]:
import numpy as np


def ising(side, beta, equilibration, sweeps, seed):
    """Mean energy and mean absolute magnetisation per spin of an Ising lattice.

    The lattice has `side` by `side` spins and wraps around at its edges. The side must be even,
    so that the two colours of the chessboard fit the wrapped edges. Every spin starts up. The
    function takes `equilibration` sweeps without measuring, and then `sweeps` sweeps, after each
    of which it measures. The random numbers come from `seed`. A routine in another language
    cannot reproduce them, so it agrees with this function only on average.
    """
    rng = np.random.default_rng(seed)
    spins = np.ones((side, side), dtype=np.int8)
    rows, columns = np.indices((side, side))
    colours = (rows + columns) % 2
    energy = magnetisation = 0.0
    for sweep in range(equilibration + sweeps):
        for colour in (0, 1):
            neighbours = (np.roll(spins, 1, axis=0) + np.roll(spins, -1, axis=0)
                          + np.roll(spins, 1, axis=1) + np.roll(spins, -1, axis=1))
            change = 2 * spins * neighbours  # the energy change if a spin were flipped
            accepted = rng.random((side, side)) < np.exp(-beta * change)
            spins = np.where((colours == colour) & accepted, -spins, spins)
        if sweep >= equilibration:
            bonds = spins * (np.roll(spins, 1, axis=0) + np.roll(spins, 1, axis=1))
            energy -= np.sum(bonds) / side**2
            magnetisation += abs(np.sum(spins)) / side**2
    return energy / sweeps, magnetisation / sweeps

In [ ]:
print("energy and magnetisation per spin at beta 0.2: %.3f %.3f" % ising(16, 0.2, 100, 200, seed=1))
print("energy and magnetisation per spin at beta 0.8: %.3f %.3f" % ising(16, 0.8, 100, 200, seed=1))

The declaration below tells the harness how to call the function and how to check a routine
against it. The results are averages of random samples, so a correct routine agrees with the
function only to within the statistical error. The tolerance is absolute. At the noisiest
setting, a lattice of side 32 at inverse temperature 0.3, the results of the function have a
standard deviation of 0.004 over 300 seeds, so the difference between two independent results
has a standard deviation of 0.006, and the tolerance of 0.03 is five times that. `make_inputs`
rounds an odd size up to an even one, takes 100 sweeps to equilibrate and 400 to measure, and
chooses an inverse temperature away from the critical value of about 0.44, where the averages
are very noisy. Now and then the inverse temperature is exactly 0 or 1.

In [ ]:
def make_inputs(size, rng):
    beta = rng.choice([0.0, 1.0, rng.uniform(0.1, 0.3), rng.uniform(0.6, 0.9)],
                      p=[0.05, 0.05, 0.45, 0.45])
    return {"side": size + size % 2, "beta": float(beta), "equilibration": 100, "sweeps": 400,
            "seed": int(rng.integers(0, 2**31))}


ROUTINE = {
    "function": ising,
    "make_inputs": make_inputs,
    "arguments": {"side": "int64", "beta": "float64", "equilibration": "int64",
                  "sweeps": "int64", "seed": "int64"},
    "results": {"energy": "float64", "magnetisation": "float64"},
    "test_sizes": [32, 64, 128],
    "benchmark_size": 512,
    "tolerance": {"energy": ("absolute", 0.03), "magnetisation": ("absolute", 0.03)},
}